In [79]:
import pandas as pd
import numpy as np

import linearmodels as lm

In [80]:
df = pd.read_csv("data/processed/merged_data.csv")

In [81]:
df.columns.tolist()

['region',
 'wynagrodzenie',
 'rok',
 'bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym',
 'ogółem',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wartość_liczbowa',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wskaźnik_precyzji',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'wyższe_ogółem_wartość_liczbowa',
 'wyższe_ogółem_wskaźnik_precyzji',
 'zasadnicze_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'zasadnicze_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'średnie_(łącznie_z_zasadniczym_zawodowym/branżowym_i_policealnym)_ogółem_wartość_liczbowa',
 'średnie_(łącznie_ze_średnim_zawodowym/branżowym_i_ogólnokształcącym)_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wskaźnik_precyzji',
 'średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'saldo_migracji_ogółem',
 'wymeldowania

In [82]:
df.columns = df.columns.str.replace(' ', '_')

In [83]:
df.columns.tolist()

['region',
 'wynagrodzenie',
 'rok',
 'bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym',
 'ogółem',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wartość_liczbowa',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wskaźnik_precyzji',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'wyższe_ogółem_wartość_liczbowa',
 'wyższe_ogółem_wskaźnik_precyzji',
 'zasadnicze_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'zasadnicze_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'średnie_(łącznie_z_zasadniczym_zawodowym/branżowym_i_policealnym)_ogółem_wartość_liczbowa',
 'średnie_(łącznie_ze_średnim_zawodowym/branżowym_i_ogólnokształcącym)_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wskaźnik_precyzji',
 'średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'saldo_migracji_ogółem',
 'wymeldowania

In [84]:
"""
MODEL PANELOWY — Fixed Effects (odpowiednik R::plm)
====================================================
Wymaga wcześniejszego przejścia przez audyt_danych_panelowych.py

Instalacja:
    pip install pandas numpy linearmodels statsmodels scipy

Uruchomienie:
    python model_panelowy.py
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS, RandomEffects
from linearmodels.panel import compare
from scipy import stats
import statsmodels.formula.api as smf

# ─────────────────────────────────────────────
# KONFIGURACJA — dostosuj do swojego pliku
# ─────────────────────────────────────────────
PLIK_DANYCH   = "data/processed/merged_data.csv"
SEP           = ","

KOLUMNA_ID    = "region"
KOLUMNA_CZAS  = "rok"

# Zmienna zależna
ZMIENNA_Y     = "bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym"

# Kluczowa determinanta (wybierz JEDNĄ zgodnie z sugestią opiekuna)
# Opcja A: inwestycje
# Opcja B: gęstość podmiotów
ZMIENNA_KLUCZ = "inwestycje_zl"
# ZMIENNA_KLUCZ = "podmiot_nowo_zarejestr_na_10_tys_ludnosci_w_wieku_produkcyjnym"

# Zmienne kontrolne — wpisz te, które przeżyły audyt korelacji
ZMIENNE_KONTROLNE = [
    "wynagrodzenie",
    "saldo_migracji_ogółem",
    "liczba_pomiotow_gospodarczych",
    "mieszkania oddane do użytkowania na 10 tys. ludności",
    # dodaj/usuń wg wyników korelacji z audytu
]
# ─────────────────────────────────────────────


In [85]:
# ══════════════════════════════════════════════
# 0. WCZYTANIE I PRZYGOTOWANIE DANYCH
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 0 — Wczytanie i przygotowanie danych panelowych")
print("=" * 65)

if PLIK_DANYCH.endswith(".xlsx"):
    df = pd.read_excel(PLIK_DANYCH)
else:
    df = pd.read_csv(PLIK_DANYCH, sep=SEP, low_memory=False)

df.columns = df.columns.str.replace(' ', '_')

# Konwersja kolumn numerycznych (polskie dane mogą mieć przecinki)
for col in [ZMIENNA_Y, ZMIENNA_KLUCZ] + ZMIENNE_KONTROLNE:
    if col in df.columns:
        df[col] = (df[col].astype(str)
                          .str.replace(",", ".", regex=False)
                          .str.replace(" ", "", regex=False))
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Rok jako integer
df[KOLUMNA_CZAS] = pd.to_numeric(df[KOLUMNA_CZAS], errors="coerce").astype("Int64")

# Usuwamy wiersze z NA w kluczowych kolumnach
cols_modelu = [KOLUMNA_ID, KOLUMNA_CZAS, ZMIENNA_Y, ZMIENNA_KLUCZ] + ZMIENNE_KONTROLNE
cols_dostepne = [c for c in cols_modelu if c in df.columns]
df_model = df[cols_dostepne].dropna()

print(f"  Obserwacje po usunięciu NA: {len(df_model):,}")
print(f"  Regiony: {df_model[KOLUMNA_ID].nunique()}")
print(f"  Lata: {sorted(df_model[KOLUMNA_CZAS].unique())}\n")


KROK 0 — Wczytanie i przygotowanie danych panelowych
  Obserwacje po usunięciu NA: 176
  Regiony: 16
  Lata: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]



In [86]:
# ══════════════════════════════════════════════
# 1. KORELACJE POOLED (wskazówka opiekuna)
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 1 — Korelacje pooled (wszystkie obs. razem, bez podziału na panele)")
print("=" * 65)
print("  Opiekun: 'jeżeli korelacja będzie niska, trudno zbudować dobry model'\n")

y_series = df_model[ZMIENNA_Y]
zmienne_do_korelacji = [ZMIENNA_KLUCZ] + [c for c in ZMIENNE_KONTROLNE if c in df_model.columns]

wyniki_korelacji = []
for zmienna in zmienne_do_korelacji:
    x = df_model[zmienna]
    mask = x.notna() & y_series.notna()
    if mask.sum() < 10:
        continue
    r, p = stats.pearsonr(x[mask], y_series[mask])
    wyniki_korelacji.append({
        "zmienna": zmienna,
        "r_pearson": round(r, 4),
        "p_value": round(p, 4),
        "istotna": "✔" if p < 0.05 else "✖",
        "ocena": ("✔✔ silna" if abs(r) >= 0.5 else
                  "✔ umiark." if abs(r) >= 0.3 else
                  "~ słaba" if abs(r) >= 0.1 else
                  "✖ brak")
    })

df_corr = pd.DataFrame(wyniki_korelacji).sort_values("r_pearson", key=abs, ascending=False)
print(df_corr.to_string(index=False))

r_klucz = df_corr.loc[df_corr["zmienna"] == ZMIENNA_KLUCZ, "r_pearson"]
if not r_klucz.empty:
    r_val = r_klucz.values[0]
    if abs(r_val) < 0.15:
        print(f"\n  ⚠ Korelacja kluczowej zmiennej z Y wynosi {r_val:.3f} — niska!")
        print("    Rozważ: inną zmienną kluczową, transformację log(), lub uzasadnienie merytoryczne.")
    else:
        print(f"\n  ✔ Korelacja kluczowej zmiennej z Y: {r_val:.3f}")
print()


KROK 1 — Korelacje pooled (wszystkie obs. razem, bez podziału na panele)
  Opiekun: 'jeżeli korelacja będzie niska, trudno zbudować dobry model'

                      zmienna  r_pearson  p_value istotna     ocena
liczba_pomiotow_gospodarczych    -0.6383   0.0000       ✔  ✔✔ silna
                wynagrodzenie    -0.5246   0.0000       ✔  ✔✔ silna
                inwestycje_zl    -0.4937   0.0000       ✔ ✔ umiark.
        saldo_migracji_ogółem    -0.2665   0.0004       ✔   ~ słaba

  ✔ Korelacja kluczowej zmiennej z Y: -0.494



In [87]:
# ══════════════════════════════════════════════
# 2. DEKLARACJA DANYCH PANELOWYCH
#    odpowiednik: pdata.frame(df, index=c("region","rok"))
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 2 — Deklaracja struktury panelowej (indeks: region × rok)")
print("=" * 65)

# linearmodels wymaga MultiIndex: (entity, time)
df_panel = df_model.copy()
df_panel = df_panel.set_index([KOLUMNA_ID, KOLUMNA_CZAS])

# Formuła po prawej stronie
rhs_zmienne = [ZMIENNA_KLUCZ] + [c for c in ZMIENNE_KONTROLNE if c in df_panel.columns]
formula_rhs = " + ".join(rhs_zmienne)
print(f"  Y  = {ZMIENNA_Y}")
print(f"  X  = {formula_rhs}\n")


KROK 2 — Deklaracja struktury panelowej (indeks: region × rok)
  Y  = bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym
  X  = inwestycje_zl + wynagrodzenie + saldo_migracji_ogółem + liczba_pomiotow_gospodarczych



In [88]:
# ══════════════════════════════════════════════
# 3. MODEL POOLED OLS (punkt odniesienia)
#    dane traktowane jak zwykła regresja, bez panelu
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 3 — Model Pooled OLS (punkt odniesienia, ignoruje strukturę panelową)")
print("=" * 65)

model_pooled = PooledOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_pooled = model_pooled.fit(cov_type="clustered", cluster_entity=True)
print(wynik_pooled.summary.tables[1])
print(f"  R²: {wynik_pooled.rsquared:.4f}\n")


KROK 3 — Model Pooled OLS (punkt odniesienia, ignoruje strukturę panelową)
                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.5323     0.9089     6.0868     0.0000      3.7382      7.3264
inwestycje_zl                  5.558e-05  6.845e-05     0.8119     0.4180  -7.954e-05      0.0002
wynagrodzenie                 -2.322e-05     0.0001    -0.1553     0.8768     -0.0003      0.0003
saldo_migracji_ogółem          6.576e-05  5.659e-05     1.1620     0.2469  -4.595e-05      0.0002
liczba_pomiotow_gospodarczych    -0.0020     0.0009    -2.3067     0.0223     -0.0037     -0.0003
  R²: 0.4747



In [89]:
# ══════════════════════════════════════════════
# 4. MODEL FE — WITHIN (efekty indywidualne)
#    odpowiednik: plm(..., model="within", effect="individual")
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 4 — Model FE Within (efekty indywidualne — różnice wewnątrz regionu)")
print("=" * 65)
print("  R: plm(Y ~ X, model='within', effect='individual')\n")

model_fe = PanelOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs} + EntityEffects",
    data=df_panel
)
wynik_fe = model_fe.fit(cov_type="clustered", cluster_entity=True)
print(wynik_fe.summary.tables[1])
print(f"  R² (within): {wynik_fe.rsquared_within:.4f}")
print(f"  R² (between): {wynik_fe.rsquared_between:.4f}")
print(f"  R² (overall): {wynik_fe.rsquared_overall:.4f}")



KROK 4 — Model FE Within (efekty indywidualne — różnice wewnątrz regionu)
  R: plm(Y ~ X, model='within', effect='individual')

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.7181     0.3349     17.073     0.0000      5.0566      6.3797
inwestycje_zl                   1.11e-05  3.379e-05     0.3283     0.7431  -5.566e-05   7.785e-05
wynagrodzenie                   5.08e-05   7.16e-05     0.7095     0.4791  -9.064e-05      0.0002
saldo_migracji_ogółem          3.834e-05  8.497e-05     0.4512     0.6525     -0.0001      0.0002
liczba_pomiotow_gospodarczych    -0.0021     0.0003    -6.6714     0.0000     -0.0028     -0.0015
  R² (within): 0.5075
  R² (between): 0.3461
  R² (overall): 0.4184


In [90]:
# ══════════════════════════════════════════════
# 5. MODEL FE — TWOWAY (efekty indywidualne + czasowe)
#    odpowiednik: plm(..., model="within", effect="twoways")
#    opiekun: "przewiduję znaczną istotność komponentu czasowego"
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 5 — Model FE TwoWay (efekty regionów + efekty lat)")
print("=" * 65)
print("  R: plm(Y ~ X + factor(rok), model='within', effect='twoways')")
print("  Opiekun: 'przewiduję znaczną istotność komponentu czasowego'\n")

model_fe_time = PanelOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs} + EntityEffects + TimeEffects",
    data=df_panel
)
wynik_fe_time = model_fe_time.fit(cov_type="clustered", cluster_entity=True)
print(wynik_fe_time.summary.tables[1])
print(f"  R² (within): {wynik_fe_time.rsquared_within:.4f}")
print(f"  R² (between): {wynik_fe_time.rsquared_between:.4f}")
print(f"  R² (overall): {wynik_fe_time.rsquared_overall:.4f}")


KROK 5 — Model FE TwoWay (efekty regionów + efekty lat)
  R: plm(Y ~ X + factor(rok), model='within', effect='twoways')
  Opiekun: 'przewiduję znaczną istotność komponentu czasowego'

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         0.4520     1.5567     0.2904     0.7719     -2.6245      3.5285
inwestycje_zl                 -1.447e-05  2.279e-05    -0.6351     0.5264   -5.95e-05   3.056e-05
wynagrodzenie                     0.0003     0.0003     1.0182     0.3103     -0.0003      0.0010
saldo_migracji_ogółem          4.512e-05  1.466e-05     3.0768     0.0025   1.614e-05    7.41e-05
liczba_pomiotow_gospodarczych -5.965e-05     0.0013    -0.0463     0.9631     -0.0026      0.0025
  R² (within): -1.6576
  R² (bet

In [91]:
# ══════════════════════════════════════════════
# 6. MODEL BETWEEN (różnice między regionami)
#    odpowiednik: plm(..., model="between")
#    opiekun: "within-between"
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 6 — Model Between (różnice między regionami, średnie w czasie)")
print("=" * 65)
print("  R: plm(Y ~ X, model='between')\n")

model_be = BetweenOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_be = model_be.fit(cov_type="robust")
print(wynik_be.summary.tables[1])
print(f"  R²: {wynik_be.rsquared:.4f}\n")



KROK 6 — Model Between (różnice między regionami, średnie w czasie)
  R: plm(Y ~ X, model='between')

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.6991     3.7836     1.5063     0.1602     -2.6286      14.027
inwestycje_zl                     0.0001     0.0002     0.6581     0.5240     -0.0003      0.0005
wynagrodzenie                    -0.0001     0.0009    -0.1348     0.8952     -0.0021      0.0019
saldo_migracji_ogółem          5.646e-05  9.617e-05     0.5871     0.5690     -0.0002      0.0003
liczba_pomiotow_gospodarczych    -0.0021     0.0011    -1.7961     0.1000     -0.0046      0.0005
  R²: 0.4569



In [92]:
# ══════════════════════════════════════════════
# 7. TEST HAUSMANA — FE vs RE
#    czy Fixed Effects są lepsze niż Random Effects?
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 7 — Test Hausmana: Fixed Effects vs Random Effects")
print("=" * 65)
print("  H0: efekty indywidualne NIE są skorelowane ze zmiennymi X → RE OK")
print("  H1: efekty indywidualne SĄ skorelowane ze zmiennymi X → FE wymagane\n")

model_re = RandomEffects.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_re = model_re.fit(cov_type="robust")

# Ręczna statystyka Hausmana
b_fe = wynik_fe.params
b_re = wynik_re.params
common = b_fe.index.intersection(b_re.index)
diff = b_fe[common] - b_re[common]

V_fe = wynik_fe.cov.loc[common, common]
V_re = wynik_re.cov.loc[common, common]
V_diff = V_fe - V_re

try:
    V_diff_inv = np.linalg.pinv(V_diff.values)
    hausman_stat = float(diff.values @ V_diff_inv @ diff.values)
    hausman_df   = len(common)
    hausman_p    = 1 - stats.chi2.cdf(hausman_stat, df=hausman_df)
    print(f"  Statystyka χ²({hausman_df}) = {hausman_stat:.4f}")
    print(f"  p-value = {hausman_p:.4f}")
    if hausman_p < 0.05:
        print("  ➤ Odrzucamy H0 — stosuj Fixed Effects (FE) ✔")
    else:
        print("  ➤ Brak podstaw do odrzucenia H0 — Random Effects mogą być OK")
except Exception as e:
    print(f"  ⚠ Nie udało się policzyć testu Hausmana: {e}")
print()


KROK 7 — Test Hausmana: Fixed Effects vs Random Effects
  H0: efekty indywidualne NIE są skorelowane ze zmiennymi X → RE OK
  H1: efekty indywidualne SĄ skorelowane ze zmiennymi X → FE wymagane

  Statystyka χ²(5) = -0.3093
  p-value = 1.0000
  ➤ Brak podstaw do odrzucenia H0 — Random Effects mogą być OK



In [93]:

# ══════════════════════════════════════════════
# 8. PORÓWNANIE MODELI
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 8 — Porównanie modeli (Pooled / FE / FE TwoWay)")
print("=" * 65)

porownanie = compare({
    "Pooled OLS":  wynik_pooled,
    "FE (entity)": wynik_fe,
    "FE (twoway)": wynik_fe_time,
    "RE":          wynik_re,
}, stars=True)
print(porownanie)

KROK 8 — Porównanie modeli (Pooled / FE / FE TwoWay)
                                                                                                                       Model Comparison                                                                                                                      
                                                                              Pooled OLS                                            FE (entity)                                            FE (twoway)                                                     RE
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                         bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym     bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym     bezrobotni_w_liczbie_ludności_w_wieku

In [ ]:
# ══════════════════════════════════════════════
# 9. PODSUMOWANIE
# ══════════════════════════════════════════════
print("\n" + "=" * 65)
print("PODSUMOWANIE DO KONSULTACJI Z OPIEKUNEM")
print("=" * 65)
print(f"""
  Kluczowa determinanta : {ZMIENNA_KLUCZ}
  Zmienna zależna       : {ZMIENNA_Y}
  Liczba obserwacji     : {len(df_model):,}

  Model rekomendowany   : FE TwoWay (efekty regionów + lat)
  Uzasadnienie          : opiekun przewiduje istotny komponent czasowy
                          (historia bezrobocia w Polsce)

  Następne kroki:
    1. Sprawdź istotność zmiennych — usuń nieistotne i przetestuj ponownie
    2. Sprawdź, czy R²(within) wzrósł po dodaniu TimeEffects
    3. Na konsultacji pokaż wynik testu Hausmana i tabelę compare()
    4. Rozważ log(Y) jeśli bezrobocie ma wysoką skośność (audyt, krok 4)
""")


PODSUMOWANIE DO KONSULTACJI Z OPIEKUNEM

  Kluczowa determinanta : inwestycje_zl
  Zmienna zależna       : bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym
  Liczba obserwacji     : 176

  Model rekomendowany   : FE TwoWay (efekty regionów + lat)
  Uzasadnienie          : opiekun przewiduje istotny komponent czasowy
                          (historia bezrobocia w Polsce)

  Następne kroki:
    1. Sprawdź istotność zmiennych — usuń nieistotne i przetestuj ponownie
    2. Sprawdź, czy R²(within) wzrósł po dodaniu TimeEffects
    3. Na konsultacji pokaż wynik testu Hausmana i tabelę compare()
    4. Rozważ log(Y) jeśli bezrobocie ma wysoką skośność (audyt, krok 4)

